<hr style="border: 6px solid#003262;" />

<div align="center">
    <img src="assets/content/images/thumbnail.png" align="center" width="20%">
</div>

<br>

# SYNTHETIC DATA GENERATION - APPLIED HOMEWORK

<br>

**About:** Hands-on exercises for generating synthetic tabular data from a seed CSV and applying classification and regression models.

**Learning Goals:** Practice bootstrapping synthetic data from a seed; fit and evaluate a classifier on synthetic data; fit and evaluate a regressor on synthetic data.

**Keywords:** synthetic data, churn, classification, regression, pandas

**Prerequisite Knowledge:** (1) Python, (2) Pandas, (3) Matplotlib, (4) `01_synthetic_data_generation_applied.ipynb`

**Target User:** Data science students who have completed the companion lesson notebook


<hr style="border: 4px solid#003262;" />

<a name='Part_table_contents' id="Part_table_contents"></a>


#### CONTENTS

> #### [PART 1: GENERATE A SYNTHETIC DATASET](#Part_1)
> #### [PART 2: CLASSIFY DONOR CHURN](#Part_2)
> #### [PART 3: GENERALIZATION](#Part_3)

<br>


In [ ]:
# All imports for this homework live in this single cell.
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# TODO: verify against current sklearn docs
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score


<a id='Part_1'></a>

<hr style="border: 2px solid#003262;" />

#### PART 1

## **GENERATE** A SYNTHETIC **DATASET**

The lesson introduced four type-aware generator functions that sample synthetic columns from a small seed distribution. Here they are provided complete so you can focus on the applied workflow: loading the seed, calling the generators, and assembling a synthetic table.

The seed file is `data_seeds/intern_dropouts_seed.csv`. Your first job is to inspect it, then produce a 300 row synthetic dataset that preserves the shape of the seed.

___

**Note:** Sources consulted for this part: pandas [`DataFrame.sample`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sample.html) and NumPy [random sampling reference](https://numpy.org/doc/stable/reference/random/index.html).

___


In [ ]:
# Pre-written generator functions carried over from the lesson.
# Do not modify - these are the building blocks Part 1 asks you to call.

def gen_bin_data(low, high, n):
    """Return a list of n integer draws from [low, high) - useful for 0/1 flags."""
    return np.random.randint(low, high, size=n).tolist()

def gen_int_data(low, high, n):
    """Return a list of n integer draws from [low, high) - useful for counts and distances."""
    return [random.randint(low, high - 1) for _ in range(n)]

def gen_flt_data(low, high, n):
    """Return a numpy array of n floats in [low, high], rounded to one decimal."""
    return np.array([round(random.uniform(low, high), 1) for _ in range(n)])

def gen_str_data(values, n):
    """Return a list of n categorical samples drawn with replacement from values."""
    return random.choices(list(values), k=n)


<a id='Part_1_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.1: LOAD THE SEED AND INSPECT

Load the seed CSV into a pandas DataFrame named `seed_df` and print its shape and the first few rows so you know which columns you are about to synthesize.


In [ ]:
seed_path = "data_seeds/intern_dropouts_seed.csv"  # don't change these lines

### YOUR CODE HERE ###
seed_df = ...

#print(seed_df.shape)
#seed_df.head()


<a id='Part_1_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.2: BUILD A 300 ROW SYNTHETIC TABLE

Use the four generator functions above to build a synthetic DataFrame named `synth_df` with 300 rows and the following columns:

- `Distance` - integers in `[1, 25)` from `gen_int_data`
- `GPA` - floats in `[2.0, 4.0]` from `gen_flt_data`
- `Class` - categorical, sampled from the unique values of `seed_df["Class"]` using `gen_str_data`
- `Low_income` - binary 0/1 from `gen_bin_data`
- `Churned` - binary 0/1 from `gen_bin_data`

Set the numpy and random seeds first so your table is reproducible.


In [ ]:
n_rows = 300           # don't change these lines
random_seed = 42       # don't change these lines

np.random.seed(random_seed)
random.seed(random_seed)

### YOUR CODE HERE ###
synth_df = pd.DataFrame({
    "Distance": ...,
    "GPA": ...,
    "Class": ...,
    "Low_income": ...,
    "Churned": ...,
})

#print(synth_df.shape)
#synth_df.head()


<a id='Part_1_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 1.3: QUICK EDA

Produce a two panel figure: a histogram of `GPA` on the left, and a bar chart of `Churned` counts on the right. This is a sanity check that the synthetic marginals look plausible before you feed the data to a model.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

### YOUR CODE HERE ###
# Left panel: histogram of synth_df["GPA"] on axes[0]
...

# Right panel: value_counts of synth_df["Churned"] as a bar chart on axes[1]
...

axes[0].set_title("GPA distribution")
axes[1].set_title("Churn class balance")
plt.tight_layout()
plt.show()


<br>

**Interpretation**

In one or two sentences, describe whether the synthetic GPA distribution looks like a plausible student population, and whether the churn labels are balanced enough for a classifier to have signal on both classes.


*ANSWER HERE*

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_2'></a>

<hr style="border: 2px solid#003262;" />

#### PART 2

## **CLASSIFY** DONOR **CHURN**

Now that you have a synthetic table, treat `Churned` as the target and everything else as features. You will encode the one categorical column, split the data, fit a Random Forest, and read the classification report to understand where the model succeeds and where it struggles.


<a id='Part_2_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.1: PREPARE FEATURES AND TARGET

Build the feature matrix `X` and target vector `y`. Encode the `Class` column using pandas `get_dummies` (this avoids assuming an ordinal relationship between Freshman/Sophomore/Junior/Senior). `y` is the `Churned` column.


In [ ]:
### YOUR CODE HERE ###
X = ...
y = ...

#print(X.shape, y.shape)
#X.head()


<a id='Part_2_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 2.2: SPLIT, FIT, AND EVALUATE

Split the data 80/20, train a `RandomForestClassifier`, generate predictions on the held out test set, and print the classification report. The random state constants below are fixed so your results align with the reference solution.


In [ ]:
test_size = 0.2        # don't change these lines
random_state = 42      # don't change these lines

### YOUR CODE HERE ###

# Split
X_train, X_test, y_train, y_test = ...

# Fit
clf = RandomForestClassifier(random_state=random_state)  # TODO: verify against current sklearn docs
...

# Predict and evaluate
y_pred = ...

#print(classification_report(y_test, y_pred))


<br>

**Interpretation**

Look at the precision, recall, and F1 for class `1` (churned). Which of those three numbers would you most want to improve if this model were used to trigger a costly outreach campaign to at-risk donors, and why?


*ANSWER HERE*

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Why does fixing a random seed matter for reproducibility of a synthetic-data pipeline, and where in this notebook would results change if you removed every seed?**

<br>

```python
# Write a short cell below that seeds numpy and random, draws 5 integers with gen_int_data,
# then seeds again with the same value and draws 5 more. Show that the two draws are identical.
```

<hr style="border: 2px solid#003262;" />


In [ ]:
### YOUR CODE HERE ###
...


*ANSWER HERE*

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->


<a id='Part_3'></a>

<hr style="border: 2px solid#003262;" />

#### PART 3

## **GENERALIZATION** TO A NEW **DOMAIN**

The intern-dropout data is one of many places the same recipe applies. To confirm the skill has transferred, you will now apply the same bootstrap-from-seed approach to a completely different domain: chemical measurements of wines. The `load_wine` bundle from scikit-learn gives you a compact numeric table that you can treat as your seed distribution.

The goal is the same three-step arc: build a synthetic table, split it, and fit a Logistic Regression classifier that predicts the wine class.


<a id='Part_3_1'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.1: LOAD THE WINE SEED

Load `load_wine(as_frame=True)` and put the feature frame in `wine_seed` and the label vector in `wine_labels`. Print the shape so you know what you are sampling from.


In [ ]:
### YOUR CODE HERE ###
wine_bundle = ...
wine_seed = ...
wine_labels = ...

#print(wine_seed.shape)
#wine_seed.head()


<a id='Part_3_2'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.2: GENERATE 200 SYNTHETIC WINE ROWS

For each feature column in `wine_seed`, draw 200 synthetic values by sampling floats between that column's observed minimum and maximum using `gen_flt_data`. For the target column, draw 200 categorical labels from the observed classes using `gen_str_data`. Assemble the result into `wine_synth` and store the labels in `wine_synth_y`.


In [ ]:
n_wine = 200         # don't change these lines
wine_seed_val = 7    # don't change these lines

np.random.seed(wine_seed_val)
random.seed(wine_seed_val)

### YOUR CODE HERE ###

# Loop feature columns and call gen_flt_data(col.min(), col.max(), n_wine)
wine_synth = pd.DataFrame({
    ...
})

# Sample labels from wine_labels.unique()
wine_synth_y = ...

#print(wine_synth.shape, len(wine_synth_y))
#wine_synth.head()


<a id='Part_3_3'></a>

<hr style="border: 1px solid#003262;" />

#### PART 3.3: TRAIN A LOGISTIC REGRESSION

Split `wine_synth` and `wine_synth_y` 80/20, fit a `LogisticRegression` classifier with `max_iter=1000`, and print the classification report plus the overall accuracy.


In [ ]:
test_size = 0.2        # don't change these lines
random_state = 0       # don't change these lines

### YOUR CODE HERE ###

# TODO: verify against current sklearn docs
wine_model = LogisticRegression(max_iter=1000)

X_train, X_test, y_train, y_test = ...
...

wine_preds = ...

#print(classification_report(y_test, wine_preds))
#print("Accuracy:", accuracy_score(y_test, wine_preds))


<br>

**Interpretation**

Because the synthetic wine rows draw features and labels independently, the label carries no real information about the features. State what you expect the accuracy to be roughly equal to for a three class problem where labels are uniformly random, and whether your observed accuracy is consistent with that expectation. What does this tell you about the limits of naive column-independent synthetic data?


*ANSWER HERE*